In [1]:
import redback
print(redback.__version__)

No module named 'lalsimulation'
lalsimulation is not installed. Some EOS based models will not work. Please use bilby eos or pass your own EOS generation class to the model
20:38 bilby INFO    : Running bilby version: 2.3.0
20:38 redback INFO    : Running redback version: 1.12.1


1.12.1


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import redback.interaction_processes as ip
import redback.sed as sed
import redback.photosphere as photosphere
from astropy.cosmology import Planck18 as cosmo 
import astropy.units as uu

import extinction
from extinction import ccm89, fitzpatrick99, apply, remove
from scipy.interpolate import RegularGridInterpolator
import sncosmo

In [15]:
#THIS IS WHERE I AM WORKING -- IT NEEDS TO WORK NOW !! 

#pre steps: import redback + other necessary packages 

#TEST -- CHANGE THE GRB TO ONE WE KNOW AND SEE IF CODE OUTPUTS THE CORRECT RESULT 

'''GRB EVENT ANALYSIS:'''
#STEP 1: Load in GRB event information eg. AB magnitude, redshift, filter, frequency of filter?, Av if applicable

#GRB: 100206A has mag = 21.7 , z = 0.4068 at 0.1315 days in R band
#GRB 060218 has mag_AB = 17.22, epoch = 11.0, z_event = 0.0331 in R-band, f_98 = 0.7 -- THIS IS WHAT WE ARE TRYING TO ACHIEVE -- DID WE ACHIEVE IT ? -- 
#where did i get this information from ? 
event_name = '060218'
AB_mag = 17.22
redshift = 0.0331
epoch = 11.0
band = 'bessellr'
#a_v = 0.39 #is this correct? change to see if flux gets brighter !! should only affect  post - extinction correction value
#WHAT WAS AV FOR 060218 ?? 
#try a fake a_v to get to answer: 
a_v = 0.13

#PUT THE CORRECT A_V MAGNITUDE ON -- IS THIS NOW WORKING ? 

#STEP 2: Convert GRB event AB magnitude into flux density using redback 
def calc_flux_density_from_ABmag(AB_mag):
    """
    Calculate flux density from AB magnitude assuming monochromatic AB filter

    :param magnitudes: AB magnitude values
    :return: flux density
    """
    return (AB_mag * uu.ABmag).to(uu.mJy)

flux_density_event = calc_flux_density_from_ABmag(AB_mag)

print("This is the flux density of  GRB event", event_name ,f"without extinction correction: {flux_density_event:.3f}")

#check this using online calculator -- CORRECT -- OK TO PROCEED TO NEXT STEP 

#STEP 3: Use fitzpatrick99 extinction function (if applicable) to find de-reddened value for GRB flux density 
#to get wavelength for GRB event, use redback tables 
#wavelength must be an array -- how to handle this: TRY: just getting a single value for wavelength in a numpy array -- WORKED !! 
wavelength =  np.array([6498.09000]) #angstroms -- r-band wavelength 
dereddened_flux_event = extinction.remove(fitzpatrick99(wavelength, a_v, 3.1),flux_density_event) 

#dereddened flux is now a 1-d array so display the first element 
print(f"This is the extinction corrected GRB event flux density found using fitzpatrick99: {dereddened_flux_event[0]:.3f}")

#to test if this is doing what it should be doing, if i increase the a_v value, the flux should go upwards!
# if Av is changed to be higher 4-5 than the original value, flux density should be getting brighter !! 
#DOES THIS HAPPEN ? -- YES -- the value goes up !!

#PROCEED TO NEXT STEP 

'''SN1998BW MODEL ANALYSIS:'''
#STEP 4: Define band from sncosmo to match the filter for the GRB event with the frequency bandpass filter for input into SN1998bw model 

print("GRB",event_name, "is associated with the filter band", band)

#we need to get frequency bandpass for the specific filter using sncosmo
bandpass = sncosmo.get_bandpass(band)
# wavelengths (in angstroms)
print(bandpass.wave)
#convert wavelength to m 
wavelength_m = bandpass.wave *1e-10
#define c
c = 3.0e8
#now to get in terms of frequencies have to calculate using f = c / wavelength
frequencies_Hz = c / wavelength_m 
#check + also do by hand -- DOES THIS WORK ? -- YES
print(frequencies_Hz)

#this is the frequency array we can now use for finding the flux_density at each time for 1998bw model 

#PROCEED TO NEXT STEP 

#STEP 5: Import the SN1998bw model from redback and set the output format to flux density

#insert model from redback: DO I NEED TO DO THIS OR CAN I JUST START USING IT IF IT IS ALREADY IN REDBACK AND I CALL IT ? 

#try and run this single line below to see if it works ! IF YES -- DONT NEED TO DEFINE, IF NO -- INCLUDE THE WHOLE DEFINITION 

#DOES THIS WORK -- NO BUT DO I EVEN NEED IT ? SEE IF IT BREAKS THE RESULT 

#from redback.templates import sn1998bw_template -- dont need to do ? 

#DO I NEED TO CHANGE lambda_obs or not ??? 
#CAN I INTERPOLATE IN THIS STEP OR DOES IT HAVE TO BE SEPARATE ? 

#IGNORE FOR NOW :
'''
#for getting singular time use a single epoch value for time
time = np.full_like(frequencies_Hz, epoch)  # epoch is your specific time value


#AI says:

 
#i need to sort my frequencies and time to both be in the same ascending / decending order
#since my frequencies were converted from wavelengths, they are decending instead of ascending -- TRUE ? -- YES 
sorted_indices = np.argsort(frequencies_Hz)
frequencies_Hz_sort = frequencies_Hz[sorted_indices]

#ensure the same applies to time -- THIS RETURNS THE TIME ARRAY TO MATCH THE SORTED ORDER OF THE FREQUENCIES ARRAY 
#BUT ARE FREQUENCY AND TIME DEPENDENT ON EACH OTHER LIKE THAT ? can i not have an arbitrary time = np.geomspace(0.01, 90, 200) ?

#BE SPECIFIC !! WE ARE NOT INTERPOLATING YET !! THIS RESULT IS PURELY JUST THE FLUX DENSITIES OF 1998BW OVER A GIVEN TIME PERIOD 
#THIS IS PRE-INTERPOLATION 
time_sort = time[sorted_indices]
'''

#RIGHT NOW: JUST TRY AN ARBITRARY TIME ARRAY AND PUT BANDS = BAND INTO SN_1998BW TEMPLATE 
#NEED TO INCLUDE FREQUENCY ? -- I HAVE INCLUDED IT -- NEEDED FROM SNCOSMO -- CORRECT ? -- 

amplitude = 1.0
'''
time = np.geomspace(0.01, 90, 200) 

flux_densities = sn1998bw_template(
    time=time,           # your array of times
    redshift=redshift,         # your redshift value -- value for 1998bw or for GRB event ? not interpolating yet ? 
    amplitude=amplitude,       # your amplitude scaling
    output_format='flux_density',
    #bands = band,
    frequency=frequencies_Hz,  # your frequency array from sncosmo
    cosmology=cosmo            # your cosmology object
)
'''
#TRIAL 1: BOTH FREQUENCY AND TIME = SAME LENGTH 
# Both arrays must be the same length, each (time[i], frequency[i]) is a pair
# APPARENTLY SED APPROACH >>> LIGHTCURVES FOR FLUX DENSITY COMPARISON 

epoch = epoch  # desired single time
#dont need a time array now since we have got a frequency array instead !! singluar time works ? -- 
#time = np.full_like(frequencies_Hz, epoch)

'''1998bwfluxfractions.ipynb
# Wrap epoch in an array
epoch_array = np.array([epoch])

flux_density_grid = sn1998bw_template(
    time=epoch_array,
    redshift=redshift,
    amplitude=amplitude,
    output_format='flux_density',
    frequency=frequencies_Hz,
    cosmology=cosmo
)
'''

#print this by displaying graph of flux density against time: 


#AI is saying i can do the interpolation all in one step with no need for np.interp 

'''# Example call for a single point
flux_1998_value = sn1998bw_template(
    time=10.5,                  # Your specific epoch
    redshift=0.05,              # Your specific redshift
    amplitude=1.0, 
    output_format='flux_density',
    frequency=4.5e14,           # The specific frequency for your filter
    cosmology=cosmo    # Optional
)

# result will be an array: [flux_value]
single_value = result[0]

print(single_value)'''

#BETTER ? -- HAS THIS WORKED ????????? ITS RUNNING NOW BUT GIVING 0.0 :/ 

# Wrap the single time in a list or array with at least two values

# OR pass your original time array if you have one.

# times = np.geomspace(0.01, 90, 200) 

# result = sn1998bw_template(
#     time=times,                 # Pass the array, not a single float
#     redshift=0.05,
#     amplitude=1.0,
#     output_format='flux_density',
#     frequency=4.5e14,           
#     cosmology=cosmo
# )

# # This will now work, and result[0] will be the flux for 10.5 days.
# single_value = result[0]

# print(single_value)

#RETURNS 0.0 -- WHY -- BECAUSE I NEED FREQUENCY TO BE AN ARRAY NO ? FIGURE THIS OUT !! 

#DE-bugging 
# print(f"Frequency requested: {frequencies_Hz[0]}")
# #dont even know what this is doing - print(f"Frequency grid range: {np.min(ff_array)} to {np.max(ff_array)}")
# print(f"Time requested range: {np.min(time)} to {np.max(time)}")
# #where is time_obs actually defined ? -- print(f"Time grid range: {np.min(time_obs)} to {np.max(time_obs)}")

#time_obs is calculated inside the function so i cant call it outside of the function unless..
#define time_obs explicitly 
# Calculate it in your main script for debugging
# redshift = 0.05
# times = np.geomspace(0.01, 90, 200)

# time_obs_debug = times * (1 + redshift) # This is the "stretched" grid

# print(f"Time requested range: {np.min(times)} to {np.max(times)}")
# print(f"Time grid range: {np.min(time_obs_debug)} to {np.max(time_obs_debug)}")


#these results reveal the exact reason i am getting zeros : 




#STEP 6: Interpolate SN1998bw model using speicifc GRB event data (redshift, epoch etc.) to get an output flux_density_at_epoch of 1998bw
#uncomment when i know flux densities in STEP 5 is working: 
# epoch = 0.1315 
# 1998_flux_at_epoch = np.interp(epoch, time, flux_densities)

# print(f"This is the interpolated flux density for SN1998bw: {1998_flux_at_epoch:.3f} at {epoch} days")


'''FINAL RATIO:'''
#STEP 7: take value from STEP 3 and STEP 6 in ratio with one another to get final result of flux ratio 

#UNCOMMENT THE FOLLOWING WHEN READY 
#f_1998bw_ratio = dereddened_flux_event / 

#print(f"The final flux density ratio result of F_GRB / F_1998bw = {f_1998bw_ratio:.3f}" )


#GOT UPTO STEP 5 !! 

#i am interpolating LIGHTCURVES not SEDs so interpolating TIME
#apparently the model definition has a failsafe so that when the frequency is a singular value but the time is an array: 
'''if isinstance(frequency, (int, float)):
frequency = np.ones_like(time) * frequency'''

#to ensure this is happening sucessfully we need to take the astropy units off the frequency ? ensuring it is simply numerical ? 


This is the flux density of  GRB event 060218 without extinction correction: 0.470 mJy
This is the extinction corrected GRB event flux density found using fitzpatrick99: 0.515 mJy
GRB 060218 is associated with the filter band bessellr
[5500. 5600. 5700. 5800. 5900. 6000. 6100. 6200. 6300. 6400. 6500. 6600.
 6700. 6800. 6900. 7000. 7100. 7200. 7300. 7400. 7500. 8000. 8500. 9000.]
[5.45454545e+14 5.35714286e+14 5.26315789e+14 5.17241379e+14
 5.08474576e+14 5.00000000e+14 4.91803279e+14 4.83870968e+14
 4.76190476e+14 4.68750000e+14 4.61538462e+14 4.54545455e+14
 4.47761194e+14 4.41176471e+14 4.34782609e+14 4.28571429e+14
 4.22535211e+14 4.16666667e+14 4.10958904e+14 4.05405405e+14
 4.00000000e+14 3.75000000e+14 3.52941176e+14 3.33333333e+14]


'if isinstance(frequency, (int, float)):\nfrequency = np.ones_like(time) * frequency'

In [17]:
#STEP 5:

#calculate 1-D array of flux densities of 1998bw over time 
#bug fix : ensure time is the same or related to the observed time 
# i was working in two different frames which caused a 0 error 

#define lambda_to_nu outside of the function: does this make the previous frequencies_Hz obsolete ? 

def lambda_to_nu(wavelength_angstrom):
    """ Converts wavelength in Angstroms to frequency in Hz """
    c = 299792458  # speed of light in m/s
    return c / (wavelength_angstrom * 1e-10)

#change the inner workings of the sn1998bw def 
def sn1998bw_template(time, redshift, amplitude, **kwargs):
    """
    A wrapper to the SN1998bw template. Only valid between 1100-11000 Angstrom and 0.01 to 90 days post explosion in rest frame

    Parameters
    ----------
    time
        time in days in observer frame (post explosion)
    redshift
        redshift
    amplitude
        amplitude scaling factor, where 1.0 is the original brightness of SN1998bw; and f_lambda is scaled by this factor
    kwargs
        Additional keyword arguments required by redback.
    frequency
        Required if output_format is 'flux_density'. frequency to calculate - Must be same length as time array or a single number).
    bands
        Required if output_format is 'magnitude' or 'flux'.
    output_format
        'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'
    cosmology
        Cosmology to use for luminosity distance calculation. Defaults to Planck18. Must be a astropy.cosmology object.
    Returns
    -------
        set by output format - 'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'

    """
    import sncosmo
    model = sncosmo.Model(source='v19-1998bw')
    original_redshift = 0.0085
    cosmology = kwargs.get("cosmology", cosmo)
    original_dl = (43*uu.Mpc).to(uu.cm).value

    # From roughly matching to Galama+ or Clocchiatti+1998bw light curves
    original_peak_time = 15
    model.set(z=original_redshift, t0=original_peak_time)
    model.set_source_peakmag(14.25, band='bessellb', magsys='ab')
    lls = np.linspace(1620, 11000, 300)
    f_lambda = model.flux(time, lls) #erg/s/cm^2/Angstrom.
    l_lambda = f_lambda * 4 * np.pi * original_dl**2  # erg/s/Angstrom

    # We consider this the rest frame spectrum of 1998bw. Now we can redshift it and scale it.
    time_obs = time * (1 + redshift)
    lambda_obs = lls * (1 + redshift)
    dl_new = cosmology.luminosity_distance(redshift).cgs.value
    f_lambda_obs = l_lambda / (4 * np.pi * dl_new**2)
    f_lambda_obs = amplitude * f_lambda_obs * (1 + redshift) # accounting for bandwidth stretching
    f_lambda_obs = f_lambda_obs * uu.erg / uu.s / uu.cm ** 2 / uu.Angstrom
    if kwargs['output_format'] == 'flux_density':
        frequency = kwargs['frequency']
        # work in obs frame
        ff_array = lambda_to_nu(lambda_obs)

        # Convert flux density to mJy
        fmjy = f_lambda_obs.to(uu.mJy, equivalencies=uu.spectral_density(wav=lambda_obs * uu.Angstrom)).value
        # Create interpolator on obs frame grid
        flux_interpolator = RegularGridInterpolator(
            (time_obs, ff_array),
            fmjy,
            bounds_error=False,
            fill_value=0.0)

        # Prepare points for interpolation
        if isinstance(frequency, (int, float)):
            frequency = np.ones_like(time) * frequency

        # Create points for evaluation
        #MODIFIED !! 
        #points = np.column_stack((time, frequency))
        # FIX: Align the query points with the grid (time_obs)
        points = np.column_stack((time_obs, frequency))

        # Return interpolated flux density with (1+z) correction for observer frame
        return flux_interpolator(points)
    elif kwargs['output_format'] == 'spectra':
        return namedtuple('output', ['time', 'lambdas', 'spectra'])(time=time_obs,
                                                                    lambdas=lambda_obs,
                                                                    spectra=f_lambda_obs)
    else:
        return sed.get_correct_output_format_from_spectra(time=time, time_eval=time_obs,
                                                          spectra=f_lambda_obs, lambda_array=lambda_obs,
                                                          **kwargs)



# Just pass an array of two values to satisfy the interpolator's grid requirement
target_epoch = epoch
times = np.array([target_epoch, target_epoch + 0.1])

#only takes in one specific frequency -- so what was the point in the sncosmo bandpass ?!!!!!! WHAT WAS IT ? -- DO WE STILL NEED IT -- 

result = sn1998bw_template(
    time=times,
    redshift=redshift,
    amplitude=1.0,
    output_format='flux_density',
    frequency=4.5e14,
    cosmology=cosmo
)

f_1998bw_interpolated = result[0]

# result[0] is now exactly the flux at day 10.5
print(f"Flux density at day {target_epoch}: {f_1998bw_interpolated} mJy")

#DOES THIS WORK WHEN I HAVE MADE THE POINTS CHANGE IN THE DEFINITION OF THE SN1998BW FUNCTION ? -- i think so ? lets change it and try for GRB which we know
 

#now finally interpolate to get the specific flux density of 1998bw at a given epoch which is the same for the GRB event: 
'''target_epoch = 0.1315  # The epoch of your specific GRB event
np.interp(x_point_to_find, x_array, y_array)
bw_flux_at_epoch = np.interp(target_epoch, times, result)
print(f"At {target_epoch} days, SN1998bw would have a flux of {bw_flux_at_epoch:.4f} mJy")'''
#DONT NEED TO DO THIS IF ONLY COMPARING THE EVENT AT ONE GIVEN EPOCH -- WHICH WE ARE !! 

#i have interpolated but havent taken the final ratio yet ! 

#SKIPPED STEP 6 -- INTERPOLATED IN ONE GO 


'''FINAL RATIO:'''
#STEP 7: take value from STEP 3 and STEP 6 in ratio with one another to get final result of flux ratio 

#UNCOMMENT THE FOLLOWING WHEN READY 
f_1998bw_ratio = dereddened_flux_event / f_1998bw_interpolated

#f_1998bw_interpolated = RATIO ANSWER !! 
#Units need to cancel out on the ratio ???????/

# Or, if it's a 1-element array
print(f"The final flux density ratio result of F_GRB / F_1998bw = {f_1998bw_ratio[0]:.3f}")

#ACTUALLY PUT IN A CLEAR SET OF VALUES THAT WORK 

Flux density at day 11.0: 0.6959939074428331 mJy
The final flux density ratio result of F_GRB / F_1998bw = 0.741 mJy


In [ ]:
#example 3: BEST SO FAR - CONTINUE WORKING FROM HERE !! 
#GRB 100206A has mag = 21.7 at 0.1315 days in R band 
#use fitzpatrick99 to convert !
#import extinction
import extinction
from extinction import ccm89, fitzpatrick99, apply, remove
from scipy.interpolate import RegularGridInterpolator
import sncosmo

band = sncosmo.get_bandpass('bessellv')
# Wavelengths (in Angstroms)
print(band.wave) 

#this needs to be the frequency bandpass from sncosmo 

#v_band_frequency = 8.58703e+13 #Hz

# Get the R-bandpass from sncosmo
band = sncosmo.get_bandpass('bessellr')
waves = band.wave  # Angstroms
trans = band.trans  # Transmission

# Convert wavelengths to frequencies (Hz)
frequencies = (c.value * 1e8) / waves  # c in m/s, Angstrom to cm
#DOES THIS WORK  ?

#find a deredened flux density for object 
wavelength = np.array([6498.09000]) #from redback, already in angstroms -- simple fix making it a np.array ? doesnt work 
Eb_v = 0.38
av = 0.38 * 3.1 #associated with specific event

#use redback to convert magnitude to flux 
magnitude = 21.7
z_event = 0.4068
band = 'bessellr'

#flux the same as flux density ? 
flux = redback.utils.bandpass_magnitude_to_flux(magnitude, band)
reddened = flux #flux density with no extinction correction (calculate from AB magnitude) 

dereddened = extinction.remove(fitzpatrick99(wavelength,av,3.1),reddened)

#TRY THIS !! 


#what does lambda to nu NOT DEFINED MEAN AGAIN ? 

def lambda_to_nu(v_wavelength):
    """
    :param wavelength: wavelength in Angstrom
    :return: frequency in Hertz
    """
    return c / (v_wavelength* 1.e-10)
    #return speed_of_light_si / (v_wavelength* 1.e-10)


#now calculate the fliux density of 1998bw atthat specific epoch
#compute magnitudes for all times for 1998bw
#change output format to flux density !! 

time = np.geomspace(0.01, 90, 200) 

#define sn1998bw template: 
def sn1998bw_template(time, redshift, amplitude, **kwargs):
    """
    A wrapper to the SN1998bw template. Only valid between 1100-11000 Angstrom and 0.01 to 90 days post explosion in rest frame

    Parameters
    ----------
    time
        time in days in observer frame (post explosion)
    redshift
        redshift
    amplitude
        amplitude scaling factor, where 1.0 is the original brightness of SN1998bw; and f_lambda is scaled by this factor
    kwargs
        Additional keyword arguments required by redback.
    frequency
        Required if output_format is 'flux_density'. frequency to calculate - Must be same length as time array or a single number).
    bands
        Required if output_format is 'magnitude' or 'flux'.
    output_format
        'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'
    cosmology
        Cosmology to use for luminosity distance calculation. Defaults to Planck18. Must be a astropy.cosmology object.
    Returns
    -------
        set by output format - 'flux_density', 'magnitude', 'spectra', 'flux', 'sncosmo_source'

    """
    import sncosmo
    model = sncosmo.Model(source='v19-1998bw')
    original_redshift = 0.0085
    cosmology = kwargs.get("cosmology", cosmo)
    original_dl = (43*uu.Mpc).to(uu.cm).value

    # From roughly matching to Galama+ or Clocchiatti+1998bw light curves
    original_peak_time = 15
    model.set(z=original_redshift, t0=original_peak_time)
    model.set_source_peakmag(14.25, band='bessellb', magsys='ab')
    lls = np.linspace(1620, 11000, 300)
    f_lambda = model.flux(time, lls) #erg/s/cm^2/Angstrom.
    l_lambda = f_lambda * 4 * np.pi * original_dl**2  # erg/s/Angstrom

    # We consider this the rest frame spectrum of 1998bw. Now we can redshift it and scale it.
    time_obs = time * (1 + redshift)
    lambda_obs = lls * (1 + redshift)
    dl_new = cosmology.luminosity_distance(redshift).cgs.value
    f_lambda_obs = l_lambda / (4 * np.pi * dl_new**2)
    f_lambda_obs = amplitude * f_lambda_obs * (1 + redshift) # accounting for bandwidth stretching
    f_lambda_obs = f_lambda_obs * uu.erg / uu.s / uu.cm ** 2 / uu.Angstrom
    if kwargs['output_format'] == 'flux_density':
        frequency = kwargs['frequency']
        # work in obs frame
        ff_array = lambda_to_nu(lambda_obs)

        # Convert flux density to mJy
        fmjy = f_lambda_obs.to(uu.mJy, equivalencies=uu.spectral_density(wav=lambda_obs * uu.Angstrom)).value
        # Create interpolator on obs frame grid
        flux_interpolator = RegularGridInterpolator(
            (time_obs, ff_array),
            fmjy,
            bounds_error=False,
            fill_value=0.0)

        # Prepare points for interpolation
        if isinstance(frequency, (int, float)):
            frequency = np.ones_like(time) * frequency

        # Create points for evaluation
        points = np.column_stack((time, frequency))

        # Return interpolated flux density with (1+z) correction for observer frame
        return flux_interpolator(points)
    elif kwargs['output_format'] == 'spectra':
        return namedtuple('output', ['time', 'lambdas', 'spectra'])(time=time_obs,
                                                                    lambdas=lambda_obs,
                                                                    spectra=f_lambda_obs)
    else:
        return sed.get_correct_output_format_from_spectra(time=time, time_eval=time_obs,
                                                          spectra=f_lambda_obs, lambda_array=lambda_obs,
                                                          **kwargs)

#need to include frequency or no ? 
fluxes = sn1998bw_template(time, redshift=z_event,
    amplitude=1.0, 
    output_format='flux_density', 
    bands=band, frequency = frequencies)

#how to combine with the other shorter function defined which didnt have this ! 

#interpolate to find magnitude at the desired epoch
#np.interp estimates the value of y at a given x positon 
epoch = 0.1315 
flux_at_epoch = np.interp(epoch, time, fluxes)

#now convert magnitude to flux (or do this already above in output fomat !!)
#check this has worked b y using a separate flux density rto magnitude converter 


#now take the dereddened flux density as a ratio with the flux density of 1998bw !! 
f_grb = dereddened 
flux_ratio = f_grb / flux_at_epoch 

print(f"At {epoch} days, the GRB is {flux_ratio:.2f} x as bright as SN 1998bw.")




In [ ]:
#solution to the code 
import sncosmo
import numpy as np
from astropy.constants import c

# Get the R-bandpass from sncosmo
band = sncosmo.get_bandpass('bessellr')

# Use the effective wavelength of the bandpass (in Angstroms)
lambda_eff = band.wave_eff  # in Angstroms

# Convert to frequency in Hz: c (cm/s) / lambda (cm)
frequency = (c.value * 1e8) / lambda_eff  # c.value is in m/s, so multiply by 1e8 to get Angstrom/s

# Now call your template function with this frequency
fluxes = sn1998bw_template(
    time,
    redshift=z_event,
    amplitude=1.0,
    output_format='flux_density',
    bands='bessellr',
    frequency=frequency
) 

In [ ]:
#READ THROUGH EXMPLANATION TO SEE IF THIS IS CORRECT

import sncosmo
import numpy as np
from astropy.constants import c

# Get the R-bandpass from sncosmo
band = sncosmo.get_bandpass('bessellr')
waves = band.wave  # Angstroms
trans = band.trans  # Transmission

# Convert wavelengths to frequencies (Hz)
frequencies = (c.value * 1e8) / waves  # c in m/s, Angstrom to cm

# Get flux density at each frequency for each time
# Ensure frequency array matches time shape if needed
fluxes = []
for t in time:
    # For each wavelength/frequency, get flux density at this time
    flux_at_waves = sn1998bw_template(
        np.array([t]),
        redshift=z_event,
        amplitude=1.0,
        output_format='flux_density',
        bands='bessellr',
        frequency=frequencies
    )
    # Integrate over the bandpass: sum(flux * transmission) / sum(transmission)
    integrated_flux = np.sum(flux_at_waves * trans) / np.sum(trans)
    fluxes.append(integrated_flux)

fluxes = np.array(fluxes)

In [ ]:
#FIGURE THIS OUT !! 

# Get the R-bandpass from sncosmo
band = sncosmo.get_bandpass('bessellr')
waves = band.wave  # Angstroms
trans = band.trans  # Transmission

# Convert wavelengths to frequencies (Hz)
frequencies = (c.value * 1e8) / waves  # c in m/s, Angstrom to cm



flux_at_waves = sn1998bw_template(
        np.array([t]),
        redshift=z_event,
        amplitude=1.0,
        output_format='flux_density',
        bands='bessellr',
        frequency=frequencies
    )
    # Integrate over the bandpass: sum(flux * transmission) / sum(transmission)
    integrated_flux = np.sum(flux_at_waves * trans) / np.sum(trans)
    fluxes.append(integrated_flux)

fluxes = np.array(fluxes)